In [0]:
%run ./01_setup_environment

In [0]:
# ========================================
# EMR & Billing Analytics Reporting
# ========================================

from pyspark.sql.functions import *

try:

    # ========================================
    # Read EMR Data
    # ========================================

    emr_df = (
        spark.read.format("csv")
        .option("header", True)
        .option("inferSchema", True)
        .load(f"{source_path}/emr")
    )

    # ========================================
    # Read Billing Transactions
    # ========================================

    billing_df = (
        spark.read.format("csv")
        .option("header", True)
        .option("inferSchema", True)
        .load(f"{source_path}/billing")
    )

    # ========================================
    # Read Clean Patient Data
    # ========================================

    patients_df = (
        spark.read.format("delta")
        .load(f"{silver_path}/patients_clean")
    )

    # ========================================
    # EMR + Patient Join
    # ========================================

    emr_report_df = emr_df.join(
        patients_df,
        "patient_id",
        "left"
    )

    # ========================================
    # Diagnosis Analytics
    # ========================================

    diagnosis_kpi_df = emr_report_df.groupBy(
        "diagnosis_code"
    ).agg(
        count("emr_id").alias("total_cases"),
        countDistinct("patient_id").alias("unique_patients")
    )

    diagnosis_kpi_df.write.format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .save(f"{gold_path}/diagnosis_analytics")

    # ========================================
    # Doctor Analytics
    # ========================================

    doctor_kpi_df = emr_report_df.groupBy(
        "doctor_id"
    ).agg(
        count("emr_id").alias("total_visits"),
        countDistinct("patient_id").alias("unique_patients")
    )

    doctor_kpi_df.write.format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .save(f"{gold_path}/doctor_analytics")

    # ========================================
    # Hospital Revenue Analytics
    # ========================================

    hospital_kpi_df = billing_df.groupBy(
        "hospital_id"
    ).agg(
        count("billing_id").alias("total_transactions"),
        countDistinct("patient_id").alias("unique_patients"),
        round(sum("billing_amount"),2).alias("total_revenue"),
        round(avg("billing_amount"),2).alias("avg_billing_amount")
    )

    hospital_kpi_df.write.format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .save(f"{gold_path}/hospital_revenue_analytics")

    # ========================================
    # Payment Status Analytics
    # ========================================

    payment_kpi_df = billing_df.groupBy(
        "payment_status"
    ).agg(
        count("billing_id").alias("total_transactions"),
        round(sum("billing_amount"),2).alias("total_amount")
    )

    payment_kpi_df.write.format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .save(f"{gold_path}/payment_status_analytics")

    print("EMR & Billing Analytics Completed Successfully")
    display(hospital_kpi_df)
    display(payment_kpi_df)

except Exception as e:
    print(f"Error occurred: {str(e)}")